# Практична робота №2. pandas: побудова DataFrame
**Дисципліна:** Аналіз даних та математичне моделювання  
**Студент:** Хрустовський Павло  
**Група:** ІТ-42  
**Варіант:** 2 (місто Львів)  

---

### Мета заняття:
Побудувати `DataFrame` з декількох масивів, закріпити роботу з індексом (`set_index`, `reset_index`), відбір рядків і стовпців через `.loc` та `.iloc`, фільтрацію за комбінованими булевими умовами та створення обчислюваних стовпців (векторизованою формулою та методом `.apply()`) на прикладі з лекційного матеріалу та навчальних даних свого варіанта.

## Демонстраційний приклад: наскрізна демонстрація (магазин)
Розберемо демонстраційний набір даних з лекції — невеликий магазин із 5 товарами (назва, ціна, залишок на складі), на якому продемонстровано базові операції бібліотеки `pandas`.

In [1]:
import pandas as pd

# Створення демонстраційного DataFrame
shop = pd.DataFrame({
    "item": ["хліб", "молоко", "яблука", "цукор", "чай"],
    "price": [28, 42, 55, 38, 65],
    "stock": [120, 60, 90, 40, 25],
})
print("Початковий DataFrame shop:")
display(shop)

# Робота з індексом
shop_by_item = shop.set_index("item")
print("\nРядок для 'молоко' через .loc:")
display(shop_by_item.loc["молоко"])
shop_by_item.reset_index(inplace=True)

# Відбір за підписом і за позицією
shop.index = range(1, 6)
print("\nshop.loc[1:3, 'price'] (підписи 1..3 включно):")
print(shop.loc[1:3, "price"])

print("\nshop.iloc[0:3, 1] (позиції 0..2, стовпець price):")
print(shop.iloc[0:3, 1])

# Фільтрація за умовою
cheap_and_stocked = shop[(shop["price"] < 50) & (shop["stock"] > 50)]
print("\nТовари з ціною < 50 та залишком > 50:")
display(cheap_and_stocked)

# Обчислювані стовпці
shop["total_value"] = shop["price"] * shop["stock"]  # векторизована формула
shop["low_stock"] = shop["stock"] < 50               # формула -> булевий стовпець

def stock_label(qty):
    if qty < 50:
        return "мало"
    elif qty < 100:
        return "достатньо"
    else:
        return "багато"

shop["stock_label"] = shop["stock"].apply(stock_label)  # .apply() для власної логіки
print("\nПідсумковий DataFrame shop:")
display(shop)

Початковий DataFrame shop:


,item,price,stock
0,хліб,28,120
1,молоко,42,60
2,яблука,55,90
3,цукор,38,40
4,чай,65,25



Рядок для 'молоко' через .loc:


price    42
stock    60
Name: молоко, dtype: int64


shop.loc[1:3, 'price'] (підписи 1..3 включно):
1    28
2    42
3    55
Name: price, dtype: int64

shop.iloc[0:3, 1] (позиції 0..2, стовпець price):
1    28
2    42
3    55
Name: price, dtype: int64

Товари з ціною < 50 та залишком > 50:


,item,price,stock
1,хліб,28,120
2,молоко,42,60



Підсумковий DataFrame shop:


,item,price,stock,total_value,low_stock,stock_label
1,хліб,28,120,3360,False,багато
2,молоко,42,60,2520,False,достатньо
3,яблука,55,90,4950,False,достатньо
4,цукор,38,40,1520,True,мало
5,чай,65,25,1625,True,мало


## Крок 1. Побудова DataFrame свого варіанта
**Варіант 2:** місто Львів.
* **Середньомісячна температура (°C)** (з Практики 1): `[-3, -2, 2, 8, 13, 16, 18, 17, 13, 8, 3, -1]`
* **Середньомісячна кількість опадів (мм)**: `[40, 38, 42, 55, 85, 100, 95, 78, 58, 48, 52, 46]`

Створимо `DataFrame`, перевіримо його перші рядки (`.head()`), розмірність (`.shape`) та типи даних стовпців (`.dtypes`).

In [2]:
months = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]
avg_temp = [-3, -2, 2, 8, 13, 16, 18, 17, 13, 8, 3, -1]
precip = [40, 38, 42, 55, 85, 100, 95, 78, 58, 48, 52, 46]

df = pd.DataFrame({
    "month": months,
    "avg_temp": avg_temp,
    "precip": precip,
})

print("Перші 5 рядків DataFrame (df.head()):")
display(df.head())

print(f"Розмірність DataFrame (df.shape): {df.shape}")
print("\nТипи даних стовпців (df.dtypes):")
print(df.dtypes)

Перші 5 рядків DataFrame (df.head()):


,month,avg_temp,precip
0,Січ,-3,40
1,Лют,-2,38
2,Бер,2,42
3,Кві,8,55
4,Тра,13,85


Розмірність DataFrame (df.shape): (12, 3)

Типи даних стовпців (df.dtypes):
month         str
avg_temp    int64
precip      int64
dtype: object


## Завдання 1. Індекс
1. Виведемо `DataFrame` з індексом за замовчуванням і переконаємось, що `df.index` — це цілочисельний діапазон від 0 до 11.
2. Замінимо індекс на стовпець `month` (`set_index("month")`) та отримаємо через `.loc` рядок для обраного місяця (наприклад, `"Тра"` — травень).
3. Повернемо звичайну нумерацію рядків за допомогою `.reset_index()`.

In [3]:
# 1. Перевірка індексу за замовчуванням
print("DataFrame зі стандартним числовим індексом (0..11):")
display(df)
print(f"Об'єкт індексу (df.index): {df.index}")
print(f"Початок: {df.index.start}, кінець: {df.index.stop}, крок: {df.index.step}, довжина: {len(df.index)}")

# 2. Встановлення стовпця month як індексу та вибірка рядка через .loc
df_by_month = df.set_index("month")
print("\nDataFrame після df.set_index('month'):")
display(df_by_month)

selected_month = "Тра"
print(f"\nРядок для місяця '{selected_month}' через .loc['{selected_month}']:")
display(df_by_month.loc[selected_month])

# 3. Повернення звичайної нумерації рядків
df_by_month.reset_index(inplace=True)
print("\nDataFrame після reset_index(inplace=True):")
display(df_by_month)

# Оновлюємо df
df = df_by_month.copy()

DataFrame зі стандартним числовим індексом (0..11):


,month,avg_temp,precip
0,Січ,-3,40
1,Лют,-2,38
2,Бер,2,42
3,Кві,8,55
4,Тра,13,85
5,Чер,16,100
6,Лип,18,95
7,Сер,17,78
8,Вер,13,58
9,Жов,8,48


Об'єкт індексу (df.index): RangeIndex(start=0, stop=12, step=1)
Початок: 0, кінець: 12, крок: 1, довжина: 12

DataFrame після df.set_index('month'):


,avg_temp,precip
month,,
Січ,-3,40
Лют,-2,38
Бер,2,42
Кві,8,55
Тра,13,85
Чер,16,100
Лип,18,95
Сер,17,78
Вер,13,58



Рядок для місяця 'Тра' через .loc['Тра']:


avg_temp    13
precip      85
Name: Тра, dtype: int64


DataFrame після reset_index(inplace=True):


,month,avg_temp,precip
0,Січ,-3,40
1,Лют,-2,38
2,Бер,2,42
3,Кві,8,55
4,Тра,13,85
5,Чер,16,100
6,Лип,18,95
7,Сер,17,78
8,Вер,13,58
9,Жов,8,48


## Завдання 2. Відбір рядків і стовпців
1. Встановимо числовий індекс від 1 до 12 (відповідно до календарного номера місяця) і через `.loc` виведемо стовпець `avg_temp` для місяців із підписом від 1 до 6.
2. Через `.iloc` виведемо той самий стовпець за позиціями 0–5 та проаналізуємо, чому результат може відрізнятись від попереднього пункту, якщо індекс не збігається з позицією.
3. Виведемо лише стовпці `month` і `precip` (без `avg_temp`) для всіх рядків.

In [4]:
# 1. Встановлення числового індексу від 1 до 12 та вибірка через .loc
df.index = range(1, 13)
print("DataFrame з числовим індексом 1..12 (номери місяців):")
display(df)

print("\n1. Стовпець avg_temp для місяців із підписом 1..6 через df.loc[1:6, 'avg_temp']:")
loc_result = df.loc[1:6, "avg_temp"]
print(loc_result)

# 2. Вибірка за позиціями 0..5 через .iloc
avg_temp_idx = df.columns.get_loc("avg_temp")
print(f"\nПозиційний номер стовпця 'avg_temp': {avg_temp_idx}")
print("2. Стовпець avg_temp за позиціями 0..5 через df.iloc[0:6, 1]:")
iloc_result = df.iloc[0:6, avg_temp_idx]
print(iloc_result)

# 3. Вибірка лише стовпців month і precip для всіх рядків
print("\n3. Лише стовпці month і precip для всіх 12 рядків:")
display(df[["month", "precip"]])

DataFrame з числовим індексом 1..12 (номери місяців):


,month,avg_temp,precip
1,Січ,-3,40
2,Лют,-2,38
3,Бер,2,42
4,Кві,8,55
5,Тра,13,85
6,Чер,16,100
7,Лип,18,95
8,Сер,17,78
9,Вер,13,58
10,Жов,8,48



1. Стовпець avg_temp для місяців із підписом 1..6 через df.loc[1:6, 'avg_temp']:
1    -3
2    -2
3     2
4     8
5    13
6    16
Name: avg_temp, dtype: int64

Позиційний номер стовпця 'avg_temp': 1
2. Стовпець avg_temp за позиціями 0..5 через df.iloc[0:6, 1]:
1    -3
2    -2
3     2
4     8
5    13
6    16
Name: avg_temp, dtype: int64

3. Лише стовпці month і precip для всіх 12 рядків:


,month,precip
1,Січ,40
2,Лют,38
3,Бер,42
4,Кві,55
5,Тра,85
6,Чер,100
7,Лип,95
8,Сер,78
9,Вер,58
10,Жов,48


> **Аналітична примітка до п. 2:**
> * У поточному стані таблиці (коли індекс задано як `1..12`), `.loc[1:6]` та `.iloc[0:6]` повертають однакові дані (рядки з січня по червень), оскільки мітки `1, 2, 3, 4, 5, 6` розташовані саме на позиціях `0, 1, 2, 3, 4, 5`.
> * Проте принципи їх роботи кардинально різні:
>   - `.loc[1:6]` орієнтується на **значення міток (labels)** індексу і зріз є **інклюзивним** з обох боків `[1, 6]`. Якби індекс починався з 0 (0..11), `.loc[1:6]` повернув би рядки з лютого по липень.
>   - `.iloc[0:6]` орієнтується виключно на **фізичні позиції (зсуви від 0)** у напіввідкритому інтервалі `[0, 6)`. Незалежно від того, якими є підписи рядків (рядки тексту, дати, числа), `.iloc[0:6]` завжди поверне перші 6 рядків.

## Завдання 3. Фільтрація за умовою
1. Виведемо місяці, де температура `avg_temp > 15`.
2. Виведемо місяці, де одночасно `avg_temp > 10` і `precip > 50` (використовуючи побітове `&` та круглі дужки).
3. Виведемо місяці, де `avg_temp < 0` або `precip > 80` (використовуючи побітове `|` та круглі дужки).

In [5]:
# 1. Місяці з avg_temp > 15
print("1. Місяці, де avg_temp > 15°C:")
cond_warm = df["avg_temp"] > 15
display(df[cond_warm])

# 2. Місяці з avg_temp > 10 та precip > 50 (через &, не and)
print("\n2. Місяці, де одночасно avg_temp > 10°C та precip > 50 мм:")
cond_warm_wet = (df["avg_temp"] > 10) & (df["precip"] > 50)
display(df[cond_warm_wet])

# 3. Місяці з avg_temp < 0 або precip > 80 (через |, не or)
print("\n3. Місяці, де avg_temp < 0°C або precip > 80 мм:")
cond_cold_or_heavy_rain = (df["avg_temp"] < 0) | (df["precip"] > 80)
display(df[cond_cold_or_heavy_rain])

1. Місяці, де avg_temp > 15°C:


,month,avg_temp,precip
6,Чер,16,100
7,Лип,18,95
8,Сер,17,78



2. Місяці, де одночасно avg_temp > 10°C та precip > 50 мм:


,month,avg_temp,precip
5,Тра,13,85
6,Чер,16,100
7,Лип,18,95
8,Сер,17,78
9,Вер,13,58



3. Місяці, де avg_temp < 0°C або precip > 80 мм:


,month,avg_temp,precip
1,Січ,-3,40
2,Лют,-2,38
5,Тра,13,85
6,Чер,16,100
7,Лип,18,95
12,Гру,-1,46


## Завдання 4. Обчислювані стовпці
1. Створимо стовпець `avg_temp_f` — середня температура у градусах Фаренгейта за векторизованою формулою: $T_F = T_C \times \frac{9}{5} + 32$.
2. Створимо булевий стовпець `is_wet` — `True`, якщо кількість опадів `precip` перевищує середнє значення `precip` по всій таблиці.
3. Створимо текстовий стовпець `season` (зима / весна / літо / осінь) за номером місяця, застосувавши метод `.apply()` із власною функцією.

In [6]:
# 1. Стовпець avg_temp_f (температура у Фаренгейтах векторизованою формулою)
df["avg_temp_f"] = df["avg_temp"] * 9 / 5 + 32

# 2. Булевий стовпець is_wet (опади більше середнього)
mean_precip = df["precip"].mean()
print(f"Середньорічна кількість опадів для м. Львів: {mean_precip:.2f} мм")
df["is_wet"] = df["precip"] > mean_precip

# 3. Текстовий стовпець season через .apply() за номером місяця
def get_season(month_num):
    """Визначає кліматичну пору року за номером місяця (1..12)."""
    if month_num in [12, 1, 2]:
        return "зима"
    elif month_num in [3, 4, 5]:
        return "весна"
    elif month_num in [6, 7, 8]:
        return "літо"
    elif month_num in [9, 10, 11]:
        return "осінь"
    return "невідомо"

# Застосовуємо до номера місяця, який наразі є числовим індексом таблиці (1..12)
df["season"] = df.index.to_series().apply(get_season)

print("\nФінальний DataFrame з усіма обчисленими стовпцями:")
display(df)

Середньорічна кількість опадів для м. Львів: 61.42 мм

Фінальний DataFrame з усіма обчисленими стовпцями:


,month,avg_temp,precip,avg_temp_f,is_wet,season
1,Січ,-3,40,26.6,False,зима
2,Лют,-2,38,28.4,False,зима
3,Бер,2,42,35.6,False,весна
4,Кві,8,55,46.4,False,весна
5,Тра,13,85,55.4,True,весна
6,Чер,16,100,60.8,True,літо
7,Лип,18,95,64.4,True,літо
8,Сер,17,78,62.6,True,літо
9,Вер,13,58,55.4,False,осінь
10,Жов,8,48,46.4,False,осінь


## Відповіді на контрольні / теоретичні запитання

### 1. Чому `.loc[1:6]` і `.iloc[0:6]` можуть повернути різну кількість рядків, залежно від того, який саме індекс у таблиці?

* **Метод `.loc`** працює виключно за **підписами (мітками, labels)** рядків і стовпців. Зріз у `.loc` є **інклюзивним з обох кінців** (`[start, stop]`), тобто включає і елемент з міткою `start`, і елемент з міткою `stop`. 
  - Якщо індекс числовий від 1 до 12, `.loc[1:6]` шукає рядки з мітками `1, 2, 3, 4, 5, 6` і повертає **6 рядків**.
  - Якщо індекс числовий від 0 до 11 (за замовчуванням), `.loc[1:6]` поверне рядки з мітками `1, 2, 3, 4, 5, 6` (тобто рядки з 2-го по 7-й у таблиці).
  - Якщо ж індекс містить нерегулярні числа, пропуски (наприклад, `[1, 3, 5, 7, ...]`), дублікати або є текстовим (`["Січ", "Лют", ...]`), результат виклику `.loc[1:6]` може містити **зовсім іншу кількість рядків** (наприклад, 3 рядки між 1 та 6) або взагалі викликати виняток `TypeError/KeyError`.
* **Метод `.iloc`** працює виключно за **фізичними порядковими номерами (integer offsets)** у нумерації з нуля. Зріз у `.iloc` є **напіввідкритим** інтервалом (`[start, stop)`), аналогічно стандартним зрізам списків у мові Python.
  - Тому вираз `.iloc[0:6]` **завжди гарантовано повертає рівно 6 перших рядків** (позиції `0, 1, 2, 3, 4, 5`), незалежно від того, як саме проіменований чи відформатований індекс таблиці.

---

### 2. Чому `df["avg_temp"] > 10 and df["precip"] > 50` викличе помилку, а `(df["avg_temp"] > 10) & (df["precip"] > 50)` — ні?

* **Оператор `and`** є стандартним логічним оператором Python. Він вимагає зведення кожного зі своїх операндів до одного скалярного булевого значення (`True` або `False`) шляхом виклику магічного методу `__bool__()`. 
  - Об'єкт `df["avg_temp"] > 10` є структурою `pandas.Series`, яка містить 12 булевих значень (масив прапорців). 
  - Оскільки для колекції з багатьох значень не існує однозначного правила (чи має бути істинним *хоча б одне* значення, чи *всі* одразу), бібліотека `pandas` викидає помилку: `ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all()`.
* **Оператор `&`** у бібліотеці `pandas` перевантажений (метод `__and__`) і виконує **поелементне (векторизоване) логічне множення (AND)** над двома об'єктами `Series`. У результаті утворюється новий булевий стовпець `Series`, де `True` стоїть лише на тих позиціях, де обидві умови виконуються одночасно.
* **Круглі дужки є обов'язковими**, оскільки в синтаксисі Python побітовий оператор `&` має **вищий пріоритет**, ніж оператори порівняння `>` або `<`. Без дужок вираз `df["avg_temp"] > 10 & df["precip"] > 50` сприймається інтерпретатором як `df["avg_temp"] > (10 & df["precip"]) > 50`, що призведе до помилки або некоректних обчислень.

---

### 3. У чому різниця між обчислюваним стовпцем, створеним формулою, і створеним через `.apply()`?

1. **Векторизація та продуктивність:**
   - **Формула** (наприклад, `df["avg_temp"] * 9 / 5 + 32` або `df["precip"] > mean_precip`) виконується як **векторизована операція**. Вона транслюється у низькорівневі скомпільовані цикли мовою C/Cython у внутрішніх структурах `NumPy`/`Pandas`. Обчислення проходять суцільним блоком у пам'яті без участі інтерпретатора Python на кожній ітерації, що забезпечує максимальну швидкість обробки (на великих датасетах — у десятки й сотні разів швидше).
   - **Метод `.apply()`** по суті реалізує **Python-цикл під капотом**, викликаючи передану функцію окремо для кожного рядка чи елемента. Це створює суттєві накладні витрати на виклик функцій інтерпретатором Python для кожного значення.
2. **Складність та гнучкість алгоритму:**
   - **Формули** чудово підходять для математичних, арифметичних та простих булевих операцій, які підтримуються векторним ядром бібліотеки.
   - **Метод `.apply()`** незамінний тоді, коли алгоритм формування значення вимагає **складної логіки**, яку важко виразити простою формулою: використання розгалужень (`if / elif / else`), обробка винятків (`try / except`), виклики сторонніх API, парсинг регулярними виразами або звернення до складних об'єктів.